# Day 4 - CAGR comparison table

Computes 1Y / 3Y / 5Y CAGR for all schemes using: 
- `Data/processed/fund_master_clean.csv`
- `Data/processed/nav_history_clean.csv`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# --- Repo-root detection (so notebook works from any working directory) ---
_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(12):
        if (cand / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (cand / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'nav_history_clean.csv').exists() and (parent / 'Data' / 'processed' / 'fund_master_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_DIR = _REPO_ROOT / 'Data' / 'processed'

nav_path = DATA_DIR / 'nav_history_clean.csv'
fund_path = DATA_DIR / 'fund_master_clean.csv'

if not nav_path.exists():
    raise FileNotFoundError(f'Missing file: {nav_path.resolve()}')
if not fund_path.exists():
    raise FileNotFoundError(f'Missing file: {fund_path.resolve()}')

nav_df = pd.read_csv(nav_path)
fund_df = pd.read_csv(fund_path)

nav_df['date'] = pd.to_datetime(nav_df['date'], errors='coerce')
nav_df['amfi_code'] = pd.to_numeric(nav_df['amfi_code'], errors='coerce').astype('Int64')
nav_df['nav'] = pd.to_numeric(nav_df['nav'], errors='coerce')

fund_df['amfi_code'] = pd.to_numeric(fund_df['amfi_code'], errors='coerce').astype('Int64')
fund_df = fund_df.dropna(subset=['amfi_code', 'scheme_name']).copy()

codes_sorted = fund_df.sort_values('amfi_code')['amfi_code'].astype(int).tolist()
amfi_to_name = dict(zip(fund_df['amfi_code'].astype(int), fund_df['scheme_name']))

print('Loaded schemes:', len(codes_sorted))
print('DATA_DIR:', DATA_DIR.resolve())

In [ ]:
# CAGR configuration
WINDOW_YEARS = {
    '1Y': 1,
    '3Y': 3,
    '5Y': 5,
}

START_DATE = pd.Timestamp('2020-01-01')
END_DATE = pd.Timestamp('2024-12-31')

def _first_nav_on_or_after(g: pd.DataFrame, target_date: pd.Timestamp):
    d = g.sort_values('date')
    d = d.loc[d['date'] >= target_date]
    if d.empty:
        return None
    return float(d.iloc[0]['nav'])

def _last_nav_on_or_before(g: pd.DataFrame, target_date: pd.Timestamp):
    d = g.sort_values('date')
    d = d.loc[d['date'] <= target_date]
    if d.empty:
        return None
    return float(d.iloc[-1]['nav'])

rows = []
for i, amfi_code in enumerate(codes_sorted, start=1):
    g = nav_df.loc[nav_df['amfi_code'] == amfi_code].copy()
    g = g.dropna(subset=['date', 'nav'])
    g = g.loc[(g['date'] >= START_DATE) & (g['date'] <= END_DATE)]
    if g.empty:
        continue

    scheme_name = amfi_to_name.get(int(amfi_code), str(amfi_code))

    nav_end = _last_nav_on_or_before(g, END_DATE)
    if nav_end is None or nav_end <= 0:
        continue

    row = {
        'amfi_code': int(amfi_code),
        'scheme_name': scheme_name,
        'end_date': END_DATE.date().isoformat(),
        'nav_end': nav_end,
    }

    for col, yrs in WINDOW_YEARS.items():
        target = END_DATE - pd.Timedelta(days=int(yrs * 365.25))
        nav_start = _first_nav_on_or_after(g, target)
        if nav_start is None or nav_start <= 0:
            row[col + '_CAGR'] = np.nan
        else:
            row[col + '_CAGR'] = (nav_end / nav_start) ** (1.0 / yrs) - 1.0

    rows.append(row)

    if i % 50 == 0:
        print(f'Processed {i}/{len(codes_sorted)} schemes...')

cagr_table = pd.DataFrame(rows)

# human-friendly formatting (percent)
for col in ['1Y_CAGR', '3Y_CAGR', '5Y_CAGR']:
    if col in cagr_table.columns:
        cagr_table[col] = cagr_table[col] * 100.0

cagr_table = cagr_table.sort_values(['5Y_CAGR', '3Y_CAGR', '1Y_CAGR'], ascending=False)
cagr_table.head(30)